In [1]:
import time
import copy
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from functools import lru_cache
from itertools import chain, product
from scipy import stats
from scipy.integrate import cumulative_trapezoid
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score, average_precision_score,
                             log_loss, precision_recall_curve)
warnings.filterwarnings('ignore')

# ==========================================
# 0. KONFIGURASI EKSPERIMEN
# ==========================================
QUICK = False                         # True = uji cepat (1 seed, epoch minimal); False = eksperimen penuh
SEEDS = [42] if QUICK else [42, 0, 1, 2, 3]
EQUIV_MARGIN = 0.5                    # margin ekuivalensi akurasi (pp). TETAPKAN SEBELUM MELIHAT HASIL
EVR_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]   # kandidat EVR untuk seleksi via validasi (tanpa oracle)
SWEEP = {"08": 0.75, "09": 1.5, "10": 2.0}                # pengali ambang MP untuk kurva trade-off
LAYER_NAMES = ["fc1", "fc2", "fc3"]
BASE_WIDTHS = (1024, 512, 256)
MAX_ROWS = 100_000
BATCH_SIZE = 256

# Dua resep pelatihan baseline: standar (seperti sebelumnya) dan teregularisasi (dropout + weight decay + early stopping)
RECIPES = {
    "standar": dict(epochs=3 if QUICK else 15, lr=0.002, weight_decay=0.0, dropout=0.0, patience=None),
    "teregularisasi": dict(epochs=6 if QUICK else 40, lr=0.002, weight_decay=1e-2, dropout=0.2, patience=5),
}
FINETUNE = dict(epochs=1 if QUICK else 3, lr=3e-4, weight_decay=0.0, patience=None)

LATENCY_BATCHES = [1, 16, 64, 256, 1024]
LATENCY_RUNS, LATENCY_WARMUP = 60, 10
LATENCY_COLS = [f"lat_{bs}" for bs in LATENCY_BATCHES]

# Dataset biner tabular dari OpenML. ID/nama BELUM diverifikasi di lingkungan saya; yang gagal dimuat dilewati otomatis.
DATASET_SPECS = {
    "Adult Income": dict(name="adult", version=2),
    "Bank Marketing": dict(data_id=1461),
    "Credit Default (Taiwan)": dict(data_id=42477),
    "HIGGS (98k)": dict(data_id=23512),
}

LABELS = {
    "00": "Baseline (Full)",
    "01": "PROPOSED: SVD + ambang MP (zero-shot)",
    "02": "Gavish-Donoho (zero-shot)",
    "03": "EVR 85% (zero-shot)",
    "04": "EVR dicocokkan ke anggaran Proposed (oracle)",
    "05": "EVR dipilih via validasi (tanpa oracle)",
    "06": "Rank seragam dicocokkan ke anggaran Proposed",
    "07": "Magnitude pruning dicocokkan ke anggaran Proposed",
    "08": "Proposed, ambang x0.75",
    "09": "Proposed, ambang x1.5",
    "10": "Proposed, ambang x2.0",
    "11": "PROPOSED + fine-tune",
    "12": "EVR oracle + fine-tune",
    "13": "EVR validasi + fine-tune",
    "14": "Rank seragam + fine-tune",
    "15": "Pruning + fine-tune",
    "16": "Proposed x1.5 + fine-tune",
    "17": "Proposed x2.0 + fine-tune",
    "18": "Model kecil dari nol (anggaran = Proposed)",
}
NAME = {code: f"{code}. {label}" for code, label in LABELS.items()}
BASELINE_NAME, PROPOSED_NAME = NAME["00"], NAME["01"]
FT_SOURCES = {"11": "01", "12": "04", "13": "05", "14": "06", "15": "07", "16": "09", "17": "10"}

TRAIN_THREADS = torch.get_num_threads()    # training memakai semua core; benchmark latensi dikunci 1 thread

# ==========================================
# 1. DATASET
# ==========================================
def load_openml_binary(spec):
    X_raw, y_raw = fetch_openml(**spec, return_X_y=True, as_frame=True)
    data = pd.concat([X_raw, y_raw.rename("__target__")], axis=1).dropna()
    if len(data) > MAX_ROWS:
        data = data.sample(MAX_ROWS, random_state=0)
    target = data.pop("__target__").astype(str)
    positive = target.value_counts().idxmin()                 # kelas minoritas = kelas positif
    X = pd.get_dummies(data, drop_first=True).astype(float).values
    return X, (target == positive).astype(int).values

def load_all_datasets():
    loaded = {}
    for name, spec in DATASET_SPECS.items():
        try:
            X, y = load_openml_binary(spec)
            loaded[name] = (X, y)
            print(f"[OK]   {name}: {len(X):,} baris, {X.shape[1]} fitur, proporsi positif {y.mean():.3f}")
        except Exception as error:
            print(f"[SKIP] {name}: gagal dimuat ({type(error).__name__}: {str(error)[:80]})")
    return loaded

def prepare_data(X, y, seed):
    """Split train/val/test per seed; val (10% dari train) untuk menala ambang F1, early stopping, dan seleksi EVR;
    scaler hanya di-fit pada data train"""
    X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.1, random_state=seed, stratify=y_trainval)
    scaler = StandardScaler().fit(X_train)

    def to_x(array):
        return torch.tensor(scaler.transform(array), dtype=torch.float32)

    return {"X_train": to_x(X_train), "y_train": torch.tensor(y_train, dtype=torch.float32).unsqueeze(1),
            "X_val": to_x(X_val), "y_val": y_val, "y_val_tensor": torch.tensor(y_val, dtype=torch.float32).unsqueeze(1),
            "X_test": to_x(X_test), "y_test": y_test}

# ==========================================
# 2. ARSITEKTUR DAN PELATIHAN
# ==========================================
class WideMLP(nn.Module):
    def __init__(self, in_dim, dropout=0.0, widths=BASE_WIDTHS):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, widths[0])
        self.fc2 = nn.Linear(widths[0], widths[1])
        self.fc3 = nn.Linear(widths[1], widths[2])
        self.head = nn.Linear(widths[2], 1)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.act(self.fc2(x)))
        x = self.drop(self.act(self.fc3(x)))
        return self.head(x)

def train_model(model, data, recipe, seed):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=recipe["lr"], weight_decay=recipe["weight_decay"])
    n_samples = len(data["X_train"])
    steps_per_epoch = n_samples // BATCH_SIZE

    # Seluruh urutan batch (semua epoch) disiapkan di muka, sehingga training cukup satu loop
    generator = torch.Generator().manual_seed(seed)
    batch_indices = torch.stack([torch.randperm(n_samples, generator=generator) for _ in range(recipe["epochs"])])
    batch_indices = batch_indices[:, :steps_per_epoch * BATCH_SIZE].reshape(-1, BATCH_SIZE)

    best_val, best_state, bad_epochs = float("inf"), None, 0
    model.train()
    for step, indices in enumerate(batch_indices):
        optimizer.zero_grad()
        criterion(model(data["X_train"][indices]), data["y_train"][indices]).backward()
        optimizer.step()

        if recipe["patience"] is not None and (step + 1) % steps_per_epoch == 0:     # early stopping via validation loss
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(data["X_val"]), data["y_val_tensor"]).item()
            model.train()
            if val_loss < best_val - 1e-4:
                best_val, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                bad_epochs += 1
            if bad_epochs >= recipe["patience"]:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

def dense_params(in_dim, widths):
    sizes = [in_dim, *widths]
    return sum(a * b + b for a, b in zip(sizes[:-1], sizes[1:])) + widths[-1] + 1

def scaled_widths(scale):
    return tuple(max(2, int(round(scale * w))) for w in BASE_WIDTHS)

def match_width(in_dim, target_params, iterations=40):
    # Lebar MLP dari nol: skala terbesar yang parameternya TIDAK melebihi anggaran Proposed
    low, high = 0.0, 1.0
    for _ in range(iterations):
        mid = (low + high) / 2
        if dense_params(in_dim, scaled_widths(mid)) > target_params:
            high = mid
        else:
            low = mid
    return low

# ==========================================
# 3. AMBANG / PEMILIHAN RANK (spektrum singular value = array numpy, menurun)
# ==========================================
@lru_cache(maxsize=None)
def mp_quantile(beta, q):
    """Kuantil ke-q distribusi Marchenko-Pastur (sigma=1, rasio aspek beta <= 1)"""
    lower, upper = (1 - np.sqrt(beta)) ** 2, (1 + np.sqrt(beta)) ** 2
    x = np.linspace(lower, upper, 20001)[1:-1]
    pdf = np.sqrt((upper - x) * (x - lower)) / (2 * np.pi * beta * x)
    cdf = cumulative_trapezoid(pdf, x, initial=0)
    return float(np.interp(q, cdf / cdf[-1], x))

def get_mp_threshold_tail(S, m, n, q=0.25):
    """[PROPOSED] Sigma dari pencocokan KUANTIL ke-q spektrum dengan kuantil MP, lalu tepi bulk sigma * (sqrt(m) + sqrt(n))"""
    small, large = sorted((m, n))
    beta = min(small / large, 0.999)
    sigma = np.quantile(S, q) / np.sqrt(large * mp_quantile(beta, q))
    return sigma * (np.sqrt(small) + np.sqrt(large))

def gavish_donoho_threshold(S, m, n):
    """[PEMBANDING] Optimal hard threshold Gavish-Donoho (2014) untuk noise tak diketahui"""
    small, large = sorted((m, n))
    beta = small / large
    return (0.56 * beta**3 - 0.95 * beta**2 + 1.82 * beta + 1.43) * np.median(S)

def get_spectra(model):
    layers = [getattr(model, name) for name in LAYER_NAMES]
    shapes = [tuple(layer.weight.shape) for layer in layers]
    spectra = [torch.linalg.svdvals(layer.weight.data).numpy() for layer in layers]
    return spectra, shapes

def ranks_by_threshold(spectra, shapes, threshold_fn, multiplier=1.0):
    return [max(1, int((S > multiplier * threshold_fn(S, m, n)).sum())) for S, (m, n) in zip(spectra, shapes)]

def ranks_evr(spectra, evr):
    return [min(len(S), int((np.cumsum(S**2) / (S**2).sum() < evr).sum()) + 1) for S in spectra]

def ranks_uniform(shapes, ratio):
    return [max(1, int(round(ratio * min(m, n)))) for m, n in shapes]

# ==========================================
# 4. KOMPRESI: SVD (RANK DIBERIKAN), MAGNITUDE PRUNING, DAN FINE-TUNING
# ==========================================
class FactoredLinear(nn.Module):
    def __init__(self, W_A, W_B, bias):
        super().__init__()
        k, in_f = W_A.shape
        out_f, _ = W_B.shape
        self.fc1 = nn.Linear(in_f, k, bias=False)
        self.fc2 = nn.Linear(k, out_f, bias=True)

        with torch.no_grad():
            self.fc1.weight.copy_(W_A)
            self.fc2.weight.copy_(W_B)
            self.fc2.bias.copy_(bias)

    def forward(self, x):
        return self.fc2(self.fc1(x))

def layer_params(m, n, k):
    # faktorisasi hanya dipakai jika menghemat parameter; selain itu layer asli dipertahankan
    return m * n + m if k * (m + n) >= m * n else k * (n + m) + m

def total_params(shapes, ranks, fixed_params):
    return sum(layer_params(m, n, k) for (m, n), k in zip(shapes, ranks)) + fixed_params

def effective_ranks_text(shapes, ranks):
    return ", ".join(f"{min(m, n) if k * (m + n) >= m * n else k}/{min(m, n)}" for (m, n), k in zip(shapes, ranks))

def compress_with_ranks(model, ranks):
    comp_net = nn.Sequential()
    for idx, (name, k) in enumerate(zip(LAYER_NAMES, ranks)):
        layer = getattr(model, name)
        W, b = layer.weight.data, layer.bias.data
        m, n = W.shape

        if k * (m + n) >= m * n:
            module = copy.deepcopy(layer)
        else:
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            sqrt_S_k = torch.diag(torch.sqrt(S[:k]))
            module = FactoredLinear(sqrt_S_k @ Vh[:k, :], U[:, :k] @ sqrt_S_k, b)

        comp_net.add_module(f"layer{idx+1}", module)
        comp_net.add_module(f"relu{idx+1}", nn.ReLU())

    comp_net.add_module("head", copy.deepcopy(model.head))
    comp_net.eval()
    return comp_net

def match_budget(rank_fn, target_params, shapes, fixed_params, iterations=30):
    # Parameter total monoton naik terhadap pengatur rank (0..1): bisection mencari nilai terbesar
    # yang TIDAK melebihi anggaran parameter Proposed
    low, high = 0.0, 1.0
    for _ in range(iterations):
        mid = (low + high) / 2
        if total_params(shapes, rank_fn(mid), fixed_params) > target_params:
            high = mid
        else:
            low = mid
    return low

def magnitude_prune(model, ratio):
    model_pruned = copy.deepcopy(model)
    with torch.no_grad():
        for name in LAYER_NAMES:
            weight = getattr(model_pruned, name).weight
            weight.mul_((weight.abs() > torch.quantile(weight.abs().flatten(), ratio)).float())
    model_pruned.eval()
    return model_pruned

def prune_ratio_for_budget(model, target_params):
    weights_total = sum(getattr(model, name).weight.numel() for name in LAYER_NAMES)
    other_params = sum(p.numel() for p in model.parameters()) - weights_total
    return float(np.clip(1 - (target_params - other_params) / weights_total, 0.0, 0.999))

def attach_masks(model):
    # Bobot terpangkas tetap nol saat fine-tuning (gradiennya dinolkan)
    for name in LAYER_NAMES:
        weight = getattr(model, name).weight
        weight.register_hook(lambda grad, mask=(weight.data != 0).float(): grad * mask)

def finetune(model, data, seed, masked):
    model_ft = copy.deepcopy(model)
    if masked:
        attach_masks(model_ft)
    train_model(model_ft, data, FINETUNE, seed)
    return model_ft

# ==========================================
# 5. EVALUASI: KUALITAS, EFISIENSI, UJI STATISTIK
# ==========================================
def get_metrics(net, x_tensor):
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(x_tensor).squeeze(1)).cpu().numpy()
    return (probs >= 0.5).astype(int), probs

def count_effective_params(net):
    return sum(int((p != 0).sum().item()) for p in net.parameters())

def count_effective_flops(net):
    # FLOPs per sampel = 2 x bobot non-nol pada semua layer Linear
    return 2 * sum(int((m.weight != 0).sum().item()) for m in net.modules() if isinstance(m, nn.Linear))

def tuned_threshold(y_val, probs_val):
    precision, recall, thresholds = precision_recall_curve(y_val, probs_val)
    f1_curve = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return thresholds[np.argmax(f1_curve)]

def val_accuracy(net, data):
    return float((get_metrics(net, data["X_val"])[0] == data["y_val"]).mean())

def select_evr_by_validation(base_model, spectra, shapes, fixed_params, data):
    # Tanpa oracle: EVR terkecil (parameter paling sedikit) yang akurasi validasinya turun <= margin dari baseline
    base_val_acc = val_accuracy(base_model, data)
    val_accs = {evr: val_accuracy(compress_with_ranks(base_model, ranks_evr(spectra, evr)), data) for evr in EVR_GRID}
    eligible = [evr for evr in EVR_GRID if val_accs[evr] >= base_val_acc - EQUIV_MARGIN / 100]
    if not eligible:
        return max(EVR_GRID)
    return min(eligible, key=lambda evr: total_params(shapes, ranks_evr(spectra, evr), fixed_params))

def timed_forward(net, x):
    start = time.perf_counter()
    net(x)
    return (time.perf_counter() - start) * 1000

def bench_median(net, x):
    net.eval()
    with torch.inference_mode():
        _ = [timed_forward(net, x) for _ in range(LATENCY_WARMUP)]
        timings = [timed_forward(net, x) for _ in range(LATENCY_RUNS)]
    return float(np.median(timings))

def measure_latencies(models, X_test):
    # Diukur dari seed pertama, single-thread, diselang-seling antar metode untuk tiap batch size
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    medians = {(bs, code): bench_median(net, X_test[:bs]) for bs, (code, net) in product(LATENCY_BATCHES, models.items())}
    torch.set_num_threads(previous_threads)
    return medians

def evaluate_variant(variant_name, net, ranks_text, seed, data, reference, latency_by_bs):
    ref_preds, ref_probs, ref_params = reference
    y_test = data["y_test"]
    preds, probs = get_metrics(net, data["X_test"])
    probs_val = get_metrics(net, data["X_val"])[1]
    threshold = tuned_threshold(data["y_val"], probs_val)       # ambang dituning di validation set, per model

    acc, acc_ref = accuracy_score(y_test, preds), accuracy_score(y_test, ref_preds)
    f1, f1_ref = f1_score(y_test, preds), f1_score(y_test, ref_preds)

    b_mc = int(((ref_preds == y_test) & (preds != y_test)).sum())
    c_mc = int(((ref_preds != y_test) & (preds == y_test)).sum())
    p_mc = 1.0 if (b_mc + c_mc) == 0 else stats.binomtest(min(b_mc, c_mc), b_mc + c_mc, 0.5).pvalue

    params = count_effective_params(net)
    return {
        "variant": variant_name, "seed": seed, "params": params, "hemat": (1 - params / ref_params) * 100,
        "flops": count_effective_flops(net), "acc": acc * 100, "f1": f1 * 100,
        "f1_tuned": f1_score(y_test, (probs >= threshold).astype(int)) * 100,
        "auc": roc_auc_score(y_test, probs) * 100, "pr_auc": average_precision_score(y_test, probs) * 100,
        "log_loss": log_loss(y_test, np.clip(probs, 1e-7, 1 - 1e-7)), "pos_rate": preds.mean() * 100,
        "d_acc": (acc - acc_ref) * 100, "d_f1": (f1 - f1_ref) * 100, "p_mcnemar": p_mc, "ranks": ranks_text,
        **{f"lat_{bs}": latency_by_bs.get(bs, np.nan) for bs in LATENCY_BATCHES},
    }

# ==========================================
# 6. SATU PENGATURAN (DATASET x SEED x RESEP BASELINE)
# ==========================================
def run_setting(dataset_name, setting, data, in_dim, seed):
    recipe = RECIPES[setting]
    base_model = WideMLP(in_dim, dropout=recipe["dropout"])
    train_model(base_model, data, recipe, seed)
    preds_base, probs_base = get_metrics(base_model, data["X_test"])
    reference = (preds_base, probs_base, count_effective_params(base_model))

    spectra, shapes = get_spectra(base_model)
    fixed_params = sum(p.numel() for p in base_model.head.parameters())

    # Anggaran parameter Proposed menjadi acuan untuk semua pembanding "anggaran sama"
    ranks_proposed = ranks_by_threshold(spectra, shapes, get_mp_threshold_tail)
    target_params = total_params(shapes, ranks_proposed, fixed_params)
    evr_oracle = match_budget(lambda v: ranks_evr(spectra, v), target_params, shapes, fixed_params)
    ratio_oracle = match_budget(lambda v: ranks_uniform(shapes, v), target_params, shapes, fixed_params)
    evr_validated = select_evr_by_validation(base_model, spectra, shapes, fixed_params, data)

    rank_plan = {
        "01": ranks_proposed,
        "02": ranks_by_threshold(spectra, shapes, gavish_donoho_threshold),
        "03": ranks_evr(spectra, 0.85),
        "04": ranks_evr(spectra, evr_oracle),
        "05": ranks_evr(spectra, evr_validated),
        "06": ranks_uniform(shapes, ratio_oracle),
        **{code: ranks_by_threshold(spectra, shapes, get_mp_threshold_tail, mult) for code, mult in SWEEP.items()},
    }
    zero_shot = {code: compress_with_ranks(base_model, ranks) for code, ranks in rank_plan.items()}
    zero_shot["07"] = magnitude_prune(base_model, prune_ratio_for_budget(base_model, target_params))
    finetuned = {code: finetune(zero_shot[source], data, seed, masked=(source == "07")) for code, source in FT_SOURCES.items()}

    small_model = WideMLP(in_dim, dropout=recipe["dropout"], widths=scaled_widths(match_width(in_dim, target_params)))
    train_model(small_model, data, recipe, seed)

    all_models = dict(sorted({"00": base_model, **zero_shot, **finetuned, "18": small_model}.items()))
    structural = {**{code: code for code in all_models}, **FT_SOURCES}        # fine-tune memakai struktur (latensi) sumbernya
    latency = measure_latencies({code: net for code, net in all_models.items() if code not in FT_SOURCES}, data["X_test"]) if seed == SEEDS[0] else {}

    ranks_text = {code: effective_ranks_text(shapes, ranks) for code, ranks in rank_plan.items()}
    ranks_text.update({code: ranks_text[source] for code, source in FT_SOURCES.items() if source in ranks_text})
    return [
        {**evaluate_variant(NAME[code], net, ranks_text.get(code, "-"), seed, data, reference,
                            {bs: latency.get((bs, structural[code]), np.nan) for bs in LATENCY_BATCHES}),
         "dataset": dataset_name, "setting": setting}
        for code, net in all_models.items()
    ]

def run_dataset_seed(dataset_name, X, y, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    data = prepare_data(X, y, seed)
    records = [run_setting(dataset_name, setting, data, X.shape[1], seed) for setting in RECIPES]
    print(f"Selesai: {dataset_name} | seed {seed}")
    return list(chain.from_iterable(records))

# ==========================================
# 7. RINGKASAN: STATISTIK, HOLM, TABEL
# ==========================================
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted, running = np.empty(len(p_values)), 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted

def holm_series(p_series, exclude):
    adjusted = p_series.copy()
    mask = ~p_series.index.isin(exclude)
    adjusted[mask] = holm_adjust(p_series[mask].values)
    return adjusted

def tost_p(delta, margin):
    # Dua uji satu-sisi (TOST) pada selisih akurasi antar seed: H1 = |selisih| < margin
    if np.std(delta) == 0:
        return 0.0 if abs(np.mean(delta)) < margin else 1.0
    p_lower = stats.ttest_1samp(delta, -margin, alternative="greater").pvalue
    p_upper = stats.ttest_1samp(delta, margin, alternative="less").pvalue
    return max(p_lower, p_upper)

def paired_p(delta):
    return 1.0 if np.std(delta) == 0 else stats.ttest_1samp(delta, 0).pvalue

def ci_text(frame, fmt):
    t_crit = stats.t.ppf(0.975, max(len(frame) - 1, 1))
    mean, half = frame.mean(), t_crit * frame.std() / np.sqrt(len(frame))
    return mean.map(fmt.format) + " [" + (mean - half).map(fmt.format) + ", " + (mean + half).map(fmt.format) + "]"

def pm(summary, mean_col, std_col, fmt="{:.2f}"):
    return summary[mean_col].map(fmt.format) + " ± " + summary[std_col].map("{:.2f}".format)

def build_tables(group):
    summary = group.groupby("variant", sort=False).agg(
        params=("params", "mean"), hemat_mean=("hemat", "mean"), hemat_std=("hemat", "std"), flops=("flops", "mean"),
        acc_mean=("acc", "mean"), acc_std=("acc", "std"), f1t_mean=("f1_tuned", "mean"), f1t_std=("f1_tuned", "std"),
        auc_mean=("auc", "mean"), auc_std=("auc", "std"), ranks=("ranks", "first"),
        **{col: (col, "first") for col in LATENCY_COLS},
    )

    def wide(column):
        return group.pivot(index="seed", columns="variant", values=column).reindex(columns=summary.index)

    d_acc, acc, log_loss_wide = wide("d_acc"), wide("acc"), wide("log_loss")
    d_log_loss = log_loss_wide.sub(log_loss_wide[BASELINE_NAME], axis=0)
    vs_proposed = acc.sub(acc[PROPOSED_NAME], axis=0)

    tost = holm_series(d_acc.apply(lambda col: tost_p(col.values, EQUIV_MARGIN)), exclude=[BASELINE_NAME])
    paired = holm_series(vs_proposed.apply(lambda col: paired_p(col.values)), exclude=[PROPOSED_NAME])
    speedup = summary[LATENCY_COLS].iloc[0] / summary[LATENCY_COLS]

    table_efficiency = pd.DataFrame({
        "Params": summary["params"].round().astype(int),
        "Hemat (%)": pm(summary, "hemat_mean", "hemat_std", "{:.1f}"),
        "FLOPs eff (K)": (summary["flops"] / 1e3).map("{:.0f}".format),
        f"Rank (seed {SEEDS[0]})": summary["ranks"],
        **{f"Spd bs={bs}": speedup[f"lat_{bs}"].map("{:.2f}x".format) for bs in LATENCY_BATCHES},
    })
    table_quality = pd.DataFrame({
        "Akurasi (%)": pm(summary, "acc_mean", "acc_std"),
        "F1@tuned (%)": pm(summary, "f1t_mean", "f1t_std"),
        "AUC (%)": pm(summary, "auc_mean", "auc_std"),
        "dAcc vs Base (pp) [CI95]": ci_text(d_acc, "{:+.2f}"),
        f"TOST p Holm (+-{EQUIV_MARGIN}pp)": tost.map("{:.4f}".format),
        "dAcc vs PROPOSED (pp)": vs_proposed.mean().map("{:+.2f}".format) + " ± " + vs_proposed.std().map("{:.2f}".format),
        "p Holm vs PROPOSED": paired.map("{:.4f}".format),
        "dLogLoss [CI95]": ci_text(d_log_loss, "{:+.4f}"),
    })
    return table_efficiency, table_quality, summary

def plot_pareto(ax, summary, title):
    codes = summary.index.str[:2].astype(int)
    styles = [("o", (codes < 11), "zero-shot"), ("^", (codes >= 11) & (codes < 18), "+ fine-tune"), ("s", (codes == 18), "kecil dari nol")]
    [ax.errorbar(summary["params"][mask], summary["acc_mean"][mask], yerr=summary["acc_std"][mask], fmt=marker, capsize=2, label=label)
     for marker, mask, label in styles]
    [ax.annotate(variant[:2], (row["params"], row["acc_mean"]), textcoords="offset points", xytext=(4, 4), fontsize=7)
     for variant, row in summary.iterrows()]
    ax.set_xscale("log")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Parameter efektif (log)")
    ax.set_ylabel("Akurasi test (%)")
    ax.legend(fontsize=7)


# ABLASI KOREKSI BIAS (mandiri). Bagian 0-7 di bawah identik dengan kompresi_tabular_multi.py (data, pelatihan, rank MP, evaluasi);
# bagian 8-9 menambahkan varian koreksi bias dan membandingkannya dengan metode usulan tanpa koreksi bias (bias-free).

# ==========================================
# 8. KOREKSI BIAS SEKUENSIAL (UNTUK ABLASI)
# ==========================================
CALIBRATION_SIZE = 1024
ABL_LABELS = {
    "00": "Baseline (Full)",
    "01": "PROPOSED bias-free (tanpa koreksi bias)",
    "02": "+ koreksi bias pra-aktivasi (batch kalibrasi)",
    "03": "+ koreksi bias pasca-ReLU (cocok mean ReLU)",
}
ABL_NAMES = {code: f"{code}. {label}" for code, label in ABL_LABELS.items()}
BIAS_MODES = {"01": None, "02": "preact", "03": "relu"}

def capture_input(model, layer, X_calibration):
    # Input NYATA ke sebuah layer (aktivasi hasil forward batch kalibrasi)
    captured = []
    handle = layer.register_forward_pre_hook(lambda module, inputs: captured.append(inputs[0].detach()))
    with torch.no_grad():
        model(X_calibration)
    handle.remove()
    return captured[0]

def solve_relu_shift(pre_activation, target_mean):
    # Geser delta per neuron sehingga mean(relu(pre + delta)) = target_mean (eksak lewat pengurutan, float64)
    pre = pre_activation.double()
    n_samples = pre.shape[0]
    sorted_pre = pre.sort(dim=0, descending=True).values
    counts = torch.arange(1, n_samples + 1, dtype=torch.float64).unsqueeze(1)
    delta = (n_samples * target_mean.double() - sorted_pre.cumsum(dim=0)) / counts
    next_pre = torch.cat([sorted_pre[1:], torch.full_like(sorted_pre[:1], -float("inf"))])
    valid = (sorted_pre + delta > 0) & (next_pre + delta <= 0)
    selected = delta.gather(0, valid.float().argmax(dim=0, keepdim=True)).squeeze(0)
    return torch.where(valid.any(dim=0), selected, -sorted_pre[0]).float()

def compress_with_bias(model, ranks, X_calibration, mode):
    # Rank sama dengan metode usulan; kompresi sekuensial: input lapisan dari jaringan yang SUDAH terkompresi di lapisan sebelumnya.
    # mode None  : b_adj = b                                   (bias-free, metode usulan)
    # mode preact: b_adj = b + W mu_asli - W_rec mu_terkompresi (cocokkan rata-rata pra-aktivasi)
    # mode relu  : b_adj = b + delta, mean(relu(.)) cocok dengan model asli (cocokkan rata-rata pasca-ReLU)
    model_compressed = copy.deepcopy(model)
    for name, k in zip(LAYER_NAMES, ranks):
        layer = getattr(model, name)
        W, b = layer.weight.data, layer.bias.data
        m, n = W.shape
        if k * (m + n) >= m * n:
            continue                                             # lapisan dipertahankan utuh (tidak menghemat parameter)
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        root = torch.sqrt(S[:k])
        W_rec = (U[:, :k] * S[:k]) @ Vh[:k, :]
        b_adj = b
        if mode is not None:
            X_orig = capture_input(model, layer, X_calibration)
            X_comp = capture_input(model_compressed, getattr(model_compressed, name), X_calibration)
            if mode == "preact":
                b_adj = b + W @ X_orig.mean(dim=0) - W_rec @ X_comp.mean(dim=0)
            else:
                b_adj = b + solve_relu_shift(X_comp @ W_rec.T + b, torch.relu(X_orig @ W.T + b).mean(dim=0))
        setattr(model_compressed, name, FactoredLinear(root.unsqueeze(1) * Vh[:k, :], U[:, :k] * root, b_adj))
    model_compressed.eval()
    return model_compressed

def run_setting_bias(dataset_name, setting, data, in_dim, seed):
    torch.manual_seed(seed)                                      # baseline rezim standar identik dengan eksperimen utama
    recipe = RECIPES[setting]
    base_model = WideMLP(in_dim, dropout=recipe["dropout"])
    train_model(base_model, data, recipe, seed)
    preds_base, probs_base = get_metrics(base_model, data["X_test"])
    reference = (preds_base, probs_base, count_effective_params(base_model))

    spectra, shapes = get_spectra(base_model)
    ranks = ranks_by_threshold(spectra, shapes, get_mp_threshold_tail)
    calibration = data["X_train"][torch.randperm(len(data["X_train"]))[:CALIBRATION_SIZE]]
    models = {"00": base_model, **{code: compress_with_bias(base_model, ranks, calibration, mode) for code, mode in BIAS_MODES.items()}}
    ranks_text = {"00": "-", **{code: effective_ranks_text(shapes, ranks) for code in BIAS_MODES}}
    return [
        {**evaluate_variant(ABL_NAMES[code], net, ranks_text[code], seed, data, reference, {}), "dataset": dataset_name, "setting": setting}
        for code, net in models.items()
    ]

def run_dataset_seed_bias(dataset_name, X, y, seed):
    np.random.seed(seed)
    data = prepare_data(X, y, seed)
    records = [run_setting_bias(dataset_name, setting, data, X.shape[1], seed) for setting in RECIPES]
    print(f"Selesai (ablasi bias): {dataset_name} | seed {seed}")
    return list(chain.from_iterable(records))

# ==========================================
# 9. RINGKASAN ABLASI (selisih berpasangan terhadap bias-free, per seed)
# ==========================================
ABL_METRICS = {"acc": ("dAkurasi (pp)", "{:+.2f}"), "f1": ("dF1@0.5 (pp)", "{:+.2f}"), "auc": ("dAUC (pp)", "{:+.2f}"),
               "log_loss": ("dLogLoss", "{:+.4f}"), "pos_rate": ("dPos-rate (pp)", "{:+.2f}")}
COMPARED = [ABL_NAMES["02"], ABL_NAMES["03"]]

def delta_cell(key, group, metric, fmt, variant):
    wide = group.pivot(index="seed", columns="variant", values=metric)
    delta = wide[variant] - wide[ABL_NAMES["01"]]
    return {"key": key, "variant": variant, "metric": metric,
            "text": ci_text(delta.to_frame(variant), fmt)[variant], "p": paired_p(delta.values)}

if __name__ == "__main__":
    datasets = load_all_datasets()
    print(f"\nAblasi koreksi bias: {len(datasets)} dataset x {len(SEEDS)} seed x {len(RECIPES)} resep (QUICK={QUICK})...")
    results = [run_dataset_seed_bias(name, X, y, seed) for (name, (X, y)), seed in product(datasets.items(), SEEDS)]
    df_records = pd.DataFrame(list(chain.from_iterable(results)))
    df_records.to_csv("ablasi_bias_per_seed.csv", index=False)

    groups = {key: g for key, g in df_records.groupby(["dataset", "setting"], sort=False)}
    cells = [delta_cell(key, groups[key], metric, fmt, variant)
             for key, (metric, (_, fmt)), variant in product(groups, ABL_METRICS.items(), COMPARED)]
    tests = pd.DataFrame(cells)
    tests[["dataset", "setting"]] = pd.DataFrame(tests["key"].tolist(), index=tests.index)
    # Holm per metrik atas seluruh pengujian (dataset x rezim x varian)
    tests["p_holm"] = tests.groupby("metric")["p"].transform(lambda s: pd.Series(holm_adjust(s.values), index=s.index))
    tests["cell"] = tests["text"] + "  p=" + tests["p_holm"].map("{:.3f}".format)
    tests.drop(columns="key").to_csv("ablasi_bias_selisih.csv", index=False)

    pd.set_option("display.width", 400)
    pd.set_option("display.max_columns", 60)
    pd.set_option("display.max_colwidth", 80)
    table_delta = tests.pivot_table(index=["dataset", "setting", "variant"], columns="metric", values="cell", aggfunc="first")[list(ABL_METRICS)]
    table_delta.columns = [ABL_METRICS[c][0] for c in table_delta.columns]
    print("\n" + "=" * 220)
    print("=== SELISIH TERHADAP BIAS-FREE (mean [CI95] antar seed; p = uji-t berpasangan, Holm per metrik atas 16 pengujian) ===")
    print("=" * 220)
    print(table_delta.to_string())

    absolute = df_records.groupby(["dataset", "setting", "variant"], sort=False)[["acc", "f1", "auc", "pos_rate"]].agg(["mean", "std"]).round(2)
    print("\n" + "=" * 100)
    print("=== NILAI ABSOLUT (mean, std antar seed; F1 pada ambang 0,5; pos-rate = % prediksi positif) ===")
    print("=" * 100)
    print(absolute.to_string())

    significant = tests[(tests["metric"] == "f1") & (tests["p_holm"] < 0.05)][["dataset", "setting", "variant", "text", "p_holm"]]
    print("\nPengujian dF1@0.5 yang signifikan setelah Holm (p < 0.05): " + ("tidak ada" if significant.empty else ""))
    print(significant.to_string(index=False) if not significant.empty else "")
    print("Catatan: baseline rezim teregularisasi dilatih ulang di skrip ini sehingga mendekati, tetapi tidak identik dengan, tabel utama; "
          "perbandingan ablasi bersifat internal (berpasangan per seed). Hasil bergantung pada protokol 72/8/20 dan 5 seed.")

[OK]   Adult Income: 45,222 baris, 97 fitur, proporsi positif 0.248
[OK]   Bank Marketing: 45,211 baris, 42 fitur, proporsi positif 0.117
[OK]   Credit Default (Taiwan): 30,000 baris, 23 fitur, proporsi positif 0.221
[OK]   HIGGS (98k): 98,049 baris, 28 fitur, proporsi positif 0.471

Ablasi koreksi bias: 4 dataset x 5 seed x 2 resep (QUICK=False)...
Selesai (ablasi bias): Adult Income | seed 42
Selesai (ablasi bias): Adult Income | seed 0
Selesai (ablasi bias): Adult Income | seed 1
Selesai (ablasi bias): Adult Income | seed 2
Selesai (ablasi bias): Adult Income | seed 3
Selesai (ablasi bias): Bank Marketing | seed 42
Selesai (ablasi bias): Bank Marketing | seed 0
Selesai (ablasi bias): Bank Marketing | seed 1
Selesai (ablasi bias): Bank Marketing | seed 2
Selesai (ablasi bias): Bank Marketing | seed 3
Selesai (ablasi bias): Credit Default (Taiwan) | seed 42
Selesai (ablasi bias): Credit Default (Taiwan) | seed 0
Selesai (ablasi bias): Credit Default (Taiwan) | seed 1
Selesai (ablasi b